In [2]:
import os
import pandas as pd
import numpy as np
import sqlalchemy
import joblib
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

# Define directories based on previous pipeline
MODELS_DIR = "models"
DATA_DIR = "data"

# Ensure output directory exists
os.makedirs(DATA_DIR, exist_ok=True)

def main():
    # ==========================================
    # 1. DATABASE CONNECTION
    # ==========================================
    db_url = os.getenv('DB_URL')
    if not db_url:
        raise ValueError("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")

    print("[SYSTEM] Connecting to the database...")
    engine = sqlalchemy.create_engine(db_url)

    # ==========================================
    # 2. LOAD DATA
    # ==========================================
    print("[SYSTEM] Loading data from view_scout_master...")
    df = pd.read_sql("SELECT * FROM view_scout_master", engine)

    # ==========================================
    # 3. FEATURE ENGINEERING
    # ==========================================
    print("[SYSTEM] Engineering features...")
    numeric_cols = [
        'age', 'total_appearances', 'international_caps', 'passport_power_rank',
        'height_in_cm', 'minutes_per_match', 'total_goals', 'total_assists',
        'goals_per_90', 'assists_per_90', 'total_yellow_cards', 'total_red_cards',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'contract_months_remaining', 'max_career_transfer_fee', 'most_recent_transfer_fee',
        'international_goals'
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['wonderkid_hype'] = np.where(df['age'] <= 22, (df['total_appearances'] + (df['international_caps'] * 3)), 0)

    conditions = [
        (df['passport_power_rank'].fillna(999) <= 15),
        (df['passport_power_rank'].fillna(999) > 15) & (df['passport_power_rank'].fillna(999) <= 60)
    ]
    df['passport_tier'] = np.select(conditions, ['Tier_1', 'Tier_2'], default='Tier_3')

    league_weights = {
        'Premier League': 1.5, 'LaLiga': 1.4, 'Serie A': 1.3, 'Bundesliga': 1.3,
        'Ligue 1': 1.2, 'Eredivisie': 1.15, 'Liga Portugal': 1.15, 'Süper Lig': 1.05
    }
    df['league_coefficient'] = df['competition_name'].map(league_weights).fillna(1.0)
    df['detailed_position'] = df['sub_position'].fillna(df['position_group']).astype(str)

    # ==========================================
    # 4. PREDICTION ROUTING
    # ==========================================
    print("[SYSTEM] Loading models...")
    
    # Load models from the MODELS_DIR
    model_full_path = os.path.join(MODELS_DIR, 'scout_model_full.pkl')
    model_perf_path = os.path.join(MODELS_DIR, 'scout_model_performance_only.pkl')
    
    if not os.path.exists(model_full_path) or not os.path.exists(model_perf_path):
        raise FileNotFoundError(f"[ERROR] Models not found in '{MODELS_DIR}'. Please run the training script first.")

    model_full = joblib.load(model_full_path)
    model_perf = joblib.load(model_perf_path)

    FULL_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
        'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
        'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'contract_months_remaining', 'wonderkid_hype', 'league_coefficient',
        'has_transfer_history', 'max_career_transfer_fee', 'most_recent_transfer_fee',
        'detailed_position', 'foot', 'passport_tier'
    ]

    PERFORMANCE_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
        'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
        'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'wonderkid_hype', 'league_coefficient', 'detailed_position', 'foot', 'passport_tier'
    ]

    print("[SYSTEM] Predicting...")
    df['predicted_value_eur'] = np.nan
    df['model_used'] = ""

    has_history_mask = df['has_transfer_history'] == 1
    no_history_mask = ~has_history_mask

    if has_history_mask.any():
        X_full = df.loc[has_history_mask, FULL_FEATURES]
        df.loc[has_history_mask, 'predicted_value_eur'] = np.expm1(model_full.predict(X_full))
        df.loc[has_history_mask, 'model_used'] = 'full'

    if no_history_mask.any():
        X_perf = df.loc[no_history_mask, PERFORMANCE_FEATURES]
        df.loc[no_history_mask, 'predicted_value_eur'] = np.expm1(model_perf.predict(X_perf))
        df.loc[no_history_mask, 'model_used'] = 'performance_only'

    df['value_gap_eur'] = df['predicted_value_eur'] - df['current_market_value']

    # ==========================================
    # 5. FILTERING GEMS
    # ==========================================
    valid_gems = df[(df['value_gap_eur'] > 0) & (df['total_appearances'] >= 15) & (df['age'] > 15)]

    def format_currency(x): return f"€{x:,.0f}"
    cols_to_format = ['current_market_value', 'predicted_value_eur', 'value_gap_eur']

    # Rename mapping for professional display
    rename_mapping = {
        'player_name': 'Player Name',
        'age': 'Age',
        'model_used': 'Model Used',
        'current_market_value': 'Current Market Value',
        'predicted_value_eur': 'Predicted Value (EUR)',
        'value_gap_eur': 'Value Gap (EUR)'
    }

    # ==========================================
    # 6. DISPLAY RESULTS AND EXPORT TO TXT
    # ==========================================
    pos_map = {
        'Goalkeeper': '🛡️ TOP 5 UNDERVALUED GOALKEEPERS',
        'Defender': '🛡️ TOP 5 UNDERVALUED DEFENDERS',
        'Attack': '🎯 TOP 5 UNDERVALUED FORWARDS',
        'Midfield': '🧠 TOP 5 UNDERVALUED MIDFIELDERS'
    }

    # We will store the output in a list and then write it to a file
    report_lines = []
    report_lines.append("ScoutAI: Undervalued Gems Report\n")
    report_lines.append("=========================================\n")

    for pos, title in pos_map.items():
        section_title = f"\n--- {title} ---"
        print(section_title)
        report_lines.append(section_title + "\n")
        
        data = valid_gems[valid_gems['position_group'] == pos].sort_values(by='value_gap_eur', ascending=False).head(5)
        
        if not data.empty:
            display_data = data[['player_name', 'age', 'model_used'] + cols_to_format].copy()
            
            # Apply currency formatting
            for col in cols_to_format:
                display_data[col] = display_data[col].map(format_currency)
            
            # Apply column renaming
            display_data = display_data.rename(columns=rename_mapping)
            
            # Print to console
            table_string = display_data.to_string(index=False)
            print(table_string)
            
            # Add to report
            report_lines.append(table_string + "\n")
        else:
            empty_msg = "No undervalued players found in this category."
            print(empty_msg)
            report_lines.append(empty_msg + "\n")

    # Save to .txt file in the DATA_DIR
    txt_export_path = os.path.join(DATA_DIR, 'undervalued_gems_report.txt')
    with open(txt_export_path, 'w', encoding='utf-8') as f:
        f.writelines(report_lines)
    
    print(f"\n[SYSTEM] Report successfully saved to '{txt_export_path}'")

if __name__ == "__main__":
    main()

[SYSTEM] Connecting to the database...
[SYSTEM] Loading data from view_scout_master...
[SYSTEM] Engineering features...
[SYSTEM] Loading models...
[SYSTEM] Predicting...

--- 🛡️ TOP 5 UNDERVALUED GOALKEEPERS ---
         Player Name  Age       Model Used Current Market Value Predicted Value (EUR) Value Gap (EUR)
Giorgi Mamardashvili 25.0             full          €28,000,000           €40,857,324     €12,857,324
     Lucas Chevalier 24.0             full          €25,000,000           €31,314,328      €6,314,328
         Luiz Júnior 25.0             full          €12,000,000           €17,639,608      €5,639,608
      Elías Ólafsson 26.0 performance_only           €3,000,000            €7,545,306      €4,545,306
              Andrew 25.0             full           €5,000,000            €8,860,697      €3,860,697

--- 🛡️ TOP 5 UNDERVALUED DEFENDERS ---
      Player Name  Age Model Used Current Market Value Predicted Value (EUR) Value Gap (EUR)
 Matthijs de Ligt 26.0       full          